# AMEX v4: Sequence Embeddings + Tabular Booster

This notebook adds representation learning on customer statement sequences, then merges learned embeddings into a tabular LightGBM model.

Runtime-first design for Colab stability:
- selected feature subset
- compact GRU autoencoder
- customer subsampling for embedding training
- optional full-run toggles

## 1. Setup
Expected Drive folder: `/content/drive/MyDrive/amex_data_parquet`

Required files:
- `train_data.parquet`
- `train_labels.csv`

In [ ]:
from __future__ import annotations

import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    HAS_TORCH = True
except Exception:
    HAS_TORCH = False

print({'lightgbm': HAS_LGB, 'torch': HAS_TORCH})
if not HAS_TORCH:
    raise RuntimeError('PyTorch is required for v4. Install torch in this runtime.')

In [ ]:
# Colab/Drive path config
BASE_DIR = Path('/content/drive/MyDrive/amex_data_parquet')
TRAIN_PATH = BASE_DIR / 'train_data.parquet'
LABELS_PATH = BASE_DIR / 'train_labels.csv'

if not TRAIN_PATH.exists() or not LABELS_PATH.exists():
    raise FileNotFoundError('Missing train_data.parquet or train_labels.csv in Drive folder.')

# Runtime-safe defaults (tune upward after successful dry run)
CFG = {
    'max_seq_len': 13,
    'embed_dim': 32,
    'hidden_dim': 64,
    'batch_size': 512,
    'epochs': 3,
    'lr': 1e-3,
    'num_workers': 2,
    'pin_memory': True,
    'train_customer_sample': 120_000,  # None for full
    'feature_set': [
        'P_2','D_39','B_1','B_2','R_1','S_3','D_41','B_3','D_44','B_4',
        'D_48','B_7','S_8','D_55','B_9','R_3','B_11','D_43','S_11','D_74'
    ],
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}
print(CFG)

## 2. Load Minimal Columns
Only sequence features + `customer_ID`, `S_2` are loaded to reduce memory pressure.

In [ ]:
use_cols = ['customer_ID', 'S_2'] + CFG['feature_set']
raw = pd.read_parquet(TRAIN_PATH, columns=use_cols)
raw['customer_ID'] = raw['customer_ID'].astype('string')
raw['S_2'] = pd.to_datetime(raw['S_2'])
labels = pd.read_csv(LABELS_PATH)
labels['customer_ID'] = labels['customer_ID'].astype('string')

print(raw.shape, labels.shape)
print('unique customers:', raw['customer_ID'].nunique())

In [ ]:
# Optional customer subsample for stable Colab training
if CFG['train_customer_sample'] is not None:
    sampled_ids = labels.sample(n=min(CFG['train_customer_sample'], len(labels)), random_state=RANDOM_STATE)['customer_ID']
    sampled_ids = sampled_ids.drop_duplicates()
    raw = raw[raw['customer_ID'].isin(sampled_ids)].copy()
    labels = labels[labels['customer_ID'].isin(sampled_ids)].copy()

print('after sampling ->', raw.shape, labels.shape, 'customers=', raw['customer_ID'].nunique())

## 3. Sequence Tensor Preparation

In [ ]:
seq_cols = [c for c in CFG['feature_set'] if c in raw.columns]
raw = raw.sort_values(['customer_ID', 'S_2']).reset_index(drop=True)

# Train medians for fill
fill_values = raw[seq_cols].median(numeric_only=True)
raw[seq_cols] = raw[seq_cols].fillna(fill_values)

# Standardize lightly for stable autoencoder optimization
means = raw[seq_cols].mean()
stds = raw[seq_cols].std().replace(0, 1.0)
raw[seq_cols] = (raw[seq_cols] - means) / stds

cust_last_date = raw.groupby('customer_ID')['S_2'].max().rename('last_date').reset_index()
model_df = labels.merge(cust_last_date, on='customer_ID', how='inner').sort_values('last_date').reset_index(drop=True)
cut = int(0.8 * len(model_df))
train_ids = set(model_df.iloc[:cut]['customer_ID'])
valid_ids = set(model_df.iloc[cut:]['customer_ID'])

print('train customers:', len(train_ids), 'valid customers:', len(valid_ids))

In [ ]:
def build_sequence_map(df: pd.DataFrame, feature_cols: list[str], max_seq_len: int) -> dict[str, np.ndarray]:
    out = {}
    for cid, g in df.groupby('customer_ID', sort=False):
        x = g[feature_cols].values.astype(np.float32)
        if len(x) > max_seq_len:
            x = x[-max_seq_len:]
        out[str(cid)] = x
    return out

seq_map = build_sequence_map(raw[['customer_ID','S_2'] + seq_cols], seq_cols, CFG['max_seq_len'])
print('sequence map size:', len(seq_map))

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, customer_ids, seq_lookup, max_seq_len):
        self.customer_ids = [str(x) for x in customer_ids if str(x) in seq_lookup]
        self.seq_lookup = seq_lookup
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.customer_ids)

    def __getitem__(self, idx):
        cid = self.customer_ids[idx]
        x = self.seq_lookup[cid]
        L, F = x.shape
        pad = np.zeros((self.max_seq_len, F), dtype=np.float32)
        mask = np.zeros((self.max_seq_len,), dtype=np.float32)
        pad[:L] = x
        mask[:L] = 1.0
        return cid, pad, mask


def collate_fn(batch):
    cids, x, m = zip(*batch)
    return list(cids), torch.tensor(np.stack(x)), torch.tensor(np.stack(m))

## 4. GRU Autoencoder (Compact)

In [ ]:
class GRUAutoEncoder(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, emb_dim: int):
        super().__init__()
        self.encoder = nn.GRU(in_dim, hidden_dim, batch_first=True)
        self.to_emb = nn.Linear(hidden_dim, emb_dim)
        self.from_emb = nn.Linear(emb_dim, hidden_dim)
        self.decoder = nn.GRU(in_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, in_dim)

    def encode(self, x):
        _, h = self.encoder(x)
        h = h[-1]
        z = self.to_emb(h)
        return z

    def forward(self, x):
        z = self.encode(x)
        h0 = torch.tanh(self.from_emb(z)).unsqueeze(0)
        dec_in = torch.zeros_like(x)
        h, _ = self.decoder(dec_in, h0)
        rec = self.out(h)
        return rec, z


def masked_mse(recon, target, mask):
    # mask: [B, T]
    diff = (recon - target) ** 2
    diff = diff.mean(dim=2)  # [B, T]
    loss = (diff * mask).sum() / (mask.sum() + 1e-8)
    return loss

In [ ]:
train_ds = SeqDataset(list(train_ids), seq_map, CFG['max_seq_len'])
valid_ds = SeqDataset(list(valid_ids), seq_map, CFG['max_seq_len'])

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                      num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'], collate_fn=collate_fn)
valid_dl = DataLoader(valid_ds, batch_size=CFG['batch_size'], shuffle=False,
                      num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'], collate_fn=collate_fn)

model = GRUAutoEncoder(in_dim=len(seq_cols), hidden_dim=CFG['hidden_dim'], emb_dim=CFG['embed_dim']).to(CFG['device'])
opt = torch.optim.Adam(model.parameters(), lr=CFG['lr'])

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    tr_loss = 0.0
    n_tr = 0
    for _, xb, mb in train_dl:
        xb = xb.to(CFG['device'])
        mb = mb.to(CFG['device'])
        opt.zero_grad()
        rec, _ = model(xb)
        loss = masked_mse(rec, xb, mb)
        loss.backward()
        opt.step()
        tr_loss += loss.item() * xb.size(0)
        n_tr += xb.size(0)

    model.eval()
    va_loss = 0.0
    n_va = 0
    with torch.no_grad():
        for _, xb, mb in valid_dl:
            xb = xb.to(CFG['device'])
            mb = mb.to(CFG['device'])
            rec, _ = model(xb)
            loss = masked_mse(rec, xb, mb)
            va_loss += loss.item() * xb.size(0)
            n_va += xb.size(0)

    print({'epoch': epoch, 'train_loss': tr_loss / max(n_tr,1), 'valid_loss': va_loss / max(n_va,1)})

## 5. Extract Customer Embeddings

In [ ]:
def encode_embeddings(model, dataset, batch_size=1024):
    dl = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                    num_workers=CFG['num_workers'], pin_memory=CFG['pin_memory'], collate_fn=collate_fn)
    rows = []
    model.eval()
    with torch.no_grad():
        for cids, xb, _ in dl:
            xb = xb.to(CFG['device'])
            z = model.encode(xb).detach().cpu().numpy()
            part = pd.DataFrame(z, columns=[f'emb_{i:02d}' for i in range(z.shape[1])])
            part.insert(0, 'customer_ID', cids)
            rows.append(part)
    return pd.concat(rows, ignore_index=True)

all_ids = labels['customer_ID'].astype('string').tolist()
all_ds = SeqDataset(all_ids, seq_map, CFG['max_seq_len'])
emb_df = encode_embeddings(model, all_ds, batch_size=CFG['batch_size'])
emb_df['customer_ID'] = emb_df['customer_ID'].astype('string')
print(emb_df.shape)
emb_df.head()

## 6. Build Lightweight Tabular Base + Merge Embeddings

In [ ]:
# Fast customer aggregates on same sequence columns
agg = raw.groupby('customer_ID')[seq_cols].agg(['last','mean','std'])
agg.columns = [f'{a}_{b}' for a,b in agg.columns]
agg = agg.reset_index()
agg['customer_ID'] = agg['customer_ID'].astype('string')

train_tbl = labels.merge(agg, on='customer_ID', how='left').merge(emb_df, on='customer_ID', how='left')
train_tbl = train_tbl.merge(cust_last_date, on='customer_ID', how='left').sort_values('last_date').reset_index(drop=True)

feature_cols = [c for c in train_tbl.columns if c not in ('customer_ID','target','last_date')]
train_tbl[feature_cols] = train_tbl[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

cut = int(0.8 * len(train_tbl))
tr = train_tbl.iloc[:cut].copy()
va = train_tbl.iloc[cut:].copy()

X_tr, y_tr = tr[feature_cols], tr['target'].astype(int)
X_va, y_va = va[feature_cols], va['target'].astype(int)
print(X_tr.shape, X_va.shape)

## 7. Train LightGBM with Embeddings

In [ ]:
if not HAS_LGB:
    raise RuntimeError('LightGBM not installed.')

params = dict(
    objective='binary',
    learning_rate=0.03,
    n_estimators=700,
    num_leaves=128,
    min_child_samples=80,
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

if CFG['device'] == 'cuda':
    params['device'] = 'gpu'

clf = lgb.LGBMClassifier(**params)
clf.fit(X_tr, y_tr)
p_va = clf.predict_proba(X_va)[:, 1]

auc = roc_auc_score(y_va, p_va)
pr_auc = average_precision_score(y_va, p_va)
print({'model': 'v4_lgb_with_embeddings', 'auc': round(float(auc), 6), 'pr_auc': round(float(pr_auc), 6)})

In [ ]:
def amex_metric_np(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    order = np.argsort(-y_pred)
    y_true_sorted = y_true[order]
    w = np.where(y_true_sorted == 0, 20, 1)

    cut = int(np.ceil(0.04 * w.sum()))
    top_mask = np.cumsum(w) <= cut
    d = (y_true_sorted[top_mask] == 1).sum() / max((y_true_sorted == 1).sum(), 1)

    def weighted_gini(a_true, a_pred):
        idx = np.argsort(-a_pred)
        a_true = a_true[idx]
        ww = np.where(a_true == 0, 20, 1)
        cum_w = np.cumsum(ww)
        cum_pos = np.cumsum(a_true * ww)
        lorentz = cum_pos / cum_pos[-1]
        return np.sum((lorentz - cum_w / cum_w[-1]) * ww)

    g = weighted_gini(y_true, y_pred) / max(weighted_gini(y_true, y_true), 1e-12)
    return 0.5 * (g + d)

print({'amex_metric_valid': round(float(amex_metric_np(y_va.values, p_va)), 6)})

## 8. Colab Stability Tips
- Start with `train_customer_sample=120000`, `epochs=2~3`.
- If runtime is stable, increase sample size before increasing model width/depth.
- Keep sequence feature count small first (20-30), then expand gradually.
- Save intermediate artifacts (`emb_df`, `train_tbl`) to parquet in Drive.